Prunner YOLOV5

In [ ]:
import argparse
import math
import os
import time
import json
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import Union

import numpy as np
import torch
import torch.nn as nn
from matplotlib import pyplot as plt

import torch_pruning as tp

from ultralytics import YOLO, __version__
from ultralytics.nn.modules import Detect, Conv, Bottleneck 
from ultralytics.nn.modules.block import C3  #  C3 blocks
from ultralytics.engine.trainer import BaseTrainer
from ultralytics.utils import YAML, LOGGER, RANK, DEFAULT_CFG_DICT, DEFAULT_CFG_KEYS
from ultralytics.utils.checks import check_yaml
from ultralytics.utils.torch_utils import initialize_weights

print("✅ All imports successful!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"torch_pruning version: {tp.__version__}")


def patch_torch_load_for_pytorch26():
    original_load = torch.load

    def patched_load(f, map_location=None, weights_only=None, **kwargs):
        try:
            return original_load(f, map_location=map_location, weights_only=False, **kwargs)
        except Exception:
            return original_load(f, map_location=map_location, weights_only=weights_only, **kwargs)

    torch.load = patched_load
    print("✅ torch.load patched for PyTorch 2.6+ compatibility")

patch_torch_load_for_pytorch26()


# ========================================
# HELPER FUNCTIONS
# ========================================

def de_parallel(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def attempt_load_one_weight(model_path, device="cpu"):
    try:
        model = YOLO(model_path)
        return model.model, None
    except Exception as e:
        print(f"❌ Could not load model: {e}")
        return None, None


def ensure_loss_device_compatibility():
    """Harmless for v5; only applies if v8DetectionLoss exists."""
    try:
        from ultralytics.utils.loss import v8DetectionLoss
    except Exception:
        return

    if getattr(v8DetectionLoss, "_device_compatibility_wrapped", False):
        return

    orig_bbox_decode = v8DetectionLoss.bbox_decode

    def bbox_decode_fixed(self, anchor_points, pred_dist):
        if hasattr(self, "proj") and isinstance(self.proj, torch.Tensor):
            if self.proj.device != pred_dist.device:
                self.proj = self.proj.to(pred_dist.device)
        return orig_bbox_decode(self, anchor_points, pred_dist)

    v8DetectionLoss.bbox_decode = bbox_decode_fixed
    v8DetectionLoss._device_compatibility_wrapped = True


def strip_optimizer_v2(f: Union[str, Path] = "best.pt", s: str = "") -> None:
    x = torch.load(f, map_location=torch.device("cpu"))
    args = {**DEFAULT_CFG_DICT, **x["train_args"]}
    if x.get("ema"):
        x["model"] = x["ema"]
    for k in "optimizer", "ema", "updates":
        x[k] = None
    for p in x["model"].parameters():
        p.requires_grad = False
    x["train_args"] = {k: v for k, v in args.items() if k in DEFAULT_CFG_KEYS}
    torch.save(x, s or f)
    mb = os.path.getsize(s or f) / 1e6
    LOGGER.info(f"Optimizer stripped from {f}, {mb:.1f}MB")


def save_model_v2(self: BaseTrainer):
    ckpt = {
        "epoch": self.epoch,
        "best_fitness": self.best_fitness,
        "model": deepcopy(de_parallel(self.model)),
        "ema": deepcopy(self.ema.ema),
        "updates": self.ema.updates,
        "optimizer": self.optimizer.state_dict(),
        "train_args": vars(self.args),
        "date": datetime.now().isoformat(),
        "version": __version__,
    }
    torch.save(ckpt, self.last)
    if self.best_fitness == self.fitness:
        torch.save(ckpt, self.best)
    if (self.epoch > 0) and (self.save_period > 0) and (self.epoch % self.save_period == 0):
        torch.save(ckpt, self.wdir / f"epoch{self.epoch}.pt")
    del ckpt


def final_eval_v2(self: BaseTrainer):
    for f in self.last, self.best:
        if f.exists():
            strip_optimizer_v2(f)
            if f is self.best:
                LOGGER.info(f"\nValidating {f}...")
                self.metrics = self.validator(model=f)
                self.metrics.pop("fitness", None)
                self.run_callbacks("on_fit_epoch_end")


def train_v2(self: YOLO, pruning=False, **kwargs):
    self._check_is_pytorch_model()
    overrides = self.overrides.copy()
    overrides.update(kwargs)
    if kwargs.get("cfg"):
        LOGGER.info(f"cfg file passed. Overriding default params with {kwargs['cfg']}.")
        overrides = YAML.load(check_yaml(kwargs["cfg"]))
    overrides["mode"] = "train"
    if not overrides.get("data"):
        raise AttributeError("Dataset required but missing, i.e. pass 'data=coco128.yaml'")
    if overrides.get("resume"):
        overrides["resume"] = self.ckpt_path

    self.task = overrides.get("task") or self.task
    self.trainer = self.task_map[self.task]["trainer"](overrides=overrides, _callbacks=self.callbacks)

    if not pruning:
        if not overrides.get("resume"):
            self.trainer.model = self.trainer.get_model(weights=self.model if self.ckpt else None, cfg=self.model.yaml)
            self.model = self.trainer.model
    else:
        self.trainer.pruning = True
        self.trainer.model = self.model
        self.trainer.save_model = save_model_v2.__get__(self.trainer)
        self.trainer.final_eval = final_eval_v2.__get__(self.trainer)

    ensure_loss_device_compatibility()
    self.trainer.hub_session = self.session
    self.trainer.train()

    if RANK in (-1, 0):
        try:
            active_device = next(self.model.parameters()).device
        except Exception:
            active_device = None
        loaded_model, _ = attempt_load_one_weight(str(self.trainer.best), device=active_device)
        if loaded_model is not None:
            self.model = loaded_model
        self.overrides = self.model.args
        self.metrics = getattr(self.trainer.validator, "metrics", None)


print("✅ Training override functions defined!")



def prepare_model_for_pruning_v5(model: nn.Module):
    """
    YOLOv5(u) in Ultralytics uses C3 blocks and is generally torch-pruning friendly out-of-the-box.
    So we do NOT replace blocks like we did for YOLOv8 (C2f -> C2f_v2).
    Keep this function in case you want to add C3-specific tweaks later.
    """
    # Optional: you can verify C3 exists
    n_c3 = sum(isinstance(m, C3) for m in model.modules())
    print(f"🔎 Found C3 blocks: {n_c3} (YOLOv5-style)")
    return model


# ========================================
# CONFIG
# ========================================

# Use yolov5nu.pt (Ultralytics-trained) OR your trained weights
MODEL_PATH = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Models\runs_yolov5\leafminer_yolov5n_ultralytics\weights\best.pt"  # <-- or r"C:\...\runs_yolov5\...\weights\best.pt"
DATA_YAML  = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\MixedDS.yaml"

ITERATIVE_STEPS = 1
TARGET_PRUNE_RATES = [0.047]
MAX_MAP_DROP = 1

pruning_cfg_template = {
    "data": DATA_YAML,
    "epochs": 100,
    "batch": 16,
    "imgsz": 640,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "exist_ok": True,
    "patience": 40,
    "save": True,
    "verbose": True,
    "plots": True,
}

all_results = {}

print("=" * 70)
print("MULTI-RATIO PRUNING EXPERIMENT (YOLOv5n/u - C3)")
print("=" * 70)
print(f"Target pruning ratios: {TARGET_PRUNE_RATES}")
print(f"Iterative steps per ratio: {ITERATIVE_STEPS}")
print(f"Total experiments: {len(TARGET_PRUNE_RATES)}")
print("=" * 70)


# ========================================
# MAIN PRUNING LOOP
# ========================================

overall_start_time = time.time()

for ratio_idx, TARGET_PRUNE_RATE in enumerate(TARGET_PRUNE_RATES):
    print("\n" + "=" * 70)
    print(f"RATIO {ratio_idx + 1}/{len(TARGET_PRUNE_RATES)}: TARGET PRUNE RATE = {TARGET_PRUNE_RATE*100:.0f}%")
    print("=" * 70)

    print("Loading model...")
    model = YOLO(MODEL_PATH)
    model.__setattr__("train_v2", train_v2.__get__(model))

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.model.to(device)
    model.model.train()
    print(f"✅ Model loaded on {device}")

    # V5 prep (no C2f replacement)
    print("Preparing YOLOv5 model for pruning...")
    prepare_model_for_pruning_v5(model.model)
    model.model.to(device)


    for _, param in model.model.named_parameters():
        param.requires_grad = True

    example_inputs = torch.randn(1, 3, pruning_cfg_template["imgsz"], pruning_cfg_template["imgsz"]).to(device)

    base_macs, base_nparams = tp.utils.count_ops_and_params(model.model, example_inputs)
    print(f"\n📊 Baseline Model:")
    print(f"  MACs: {base_macs/1e9:.3f} G")
    print(f"  Parameters: {base_nparams:,} ({base_nparams/1e6:.2f} M)")

    macs_list = [base_macs]
    nparams_list = [100]
    map_list, map50_list = [], []

    print("\nRunning baseline validation...")
    pruning_cfg = pruning_cfg_template.copy()
    pruning_cfg["name"] = f"ratio_{TARGET_PRUNE_RATE*100:.0f}_baseline_val"
    pruning_cfg["project"] = f"YOLOv5_pruning_ratio_{TARGET_PRUNE_RATE*100:.0f}"
    pruning_cfg["batch"] = 16

    validation_model = deepcopy(model)
    metric = validation_model.val(**pruning_cfg)
    init_map = metric.box.map
    init_map50 = metric.box.map50
    map_list.append(init_map)
    map50_list.append(init_map50)

    print(f" Baseline Results:")
    print(f"  mAP50-95: {init_map:.4f}")
    print(f"  mAP50:    {init_map50:.4f}")

    pruning_ratio = 1 - math.pow((1 - TARGET_PRUNE_RATE), 1 / ITERATIVE_STEPS)

    print(f"\n Pruning Configuration:")
    print(f"  Target prune rate: {TARGET_PRUNE_RATE*100:.0f}%")
    print(f"  Per-step ratio:    {pruning_ratio:.4f} ({pruning_ratio*100:.2f}%)")

    for i in range(ITERATIVE_STEPS):
        print(f"\n  Iteration {i+1}/{ITERATIVE_STEPS}")

        model.model.train()
        for _, param in model.model.named_parameters():
            param.requires_grad = True

        # Ignore Detect head only
        ignored_layers = [m for m in model.model.modules() if isinstance(m, Detect)]

        pruner = tp.pruner.GroupNormPruner(
            model.model,
            example_inputs,
            importance=tp.importance.RandomImportance(), 
            iterative_steps=1,
            pruning_ratio=pruning_ratio,
            ignored_layers=ignored_layers,
        )

        print(f"  Pruning {pruning_ratio*100:.2f}% of channels...")
        pruner.step()

        # Pre-finetune val
        pruning_cfg["name"] = f"ratio_{TARGET_PRUNE_RATE*100:.0f}_step_{i}_pre_val"
        validation_model.model = deepcopy(model.model)
        metric = validation_model.val(**pruning_cfg)
        pruned_map = metric.box.map

        pruned_macs, pruned_nparams = tp.utils.count_ops_and_params(pruner.model, example_inputs)
        current_speed_up = float(macs_list[0]) / pruned_macs

        print(f"  After Pruning:")
        print(f"    MACs:   {pruned_macs/1e9:.3f} G (speedup: {current_speed_up:.2f}x)")
        print(f"    Params: {pruned_nparams/1e6:.3f} M ({pruned_nparams/base_nparams*100:.1f}% remaining)")
        print(f"    mAP (before fine-tune): {pruned_map:.4f}")


        print(f"  Fine-tuning for {pruning_cfg_template['epochs']} epochs...")
        for _, param in model.model.named_parameters():
            param.requires_grad = True

        pruning_cfg["name"] = f"ratio_{TARGET_PRUNE_RATE*100:.0f}_step_{i}_finetune"
        pruning_cfg["batch"] = 16
        model.train_v2(pruning=True, **pruning_cfg)

        pruning_cfg["name"] = f"ratio_{TARGET_PRUNE_RATE*100:.0f}_step_{i}_post_val"
        pruning_cfg["batch"] = 16
        validation_model = YOLO(model.trainer.best)
        metric = validation_model.val(**pruning_cfg)
        current_map = metric.box.map
        current_map50 = metric.box.map50

        print(f"  After Fine-tuning:")
        print(f"    mAP50-95: {current_map:.4f}")
        print(f"    mAP50:    {current_map50:.4f}")

        macs_list.append(pruned_macs)
        nparams_list.append(pruned_nparams / base_nparams * 100)
        map_list.append(current_map)
        map50_list.append(current_map50)

        del pruner

        if init_map - current_map > MAX_MAP_DROP:
            print("Early stopping, mAP drop exceeds threshold")
            break

    final_map = map_list[-1]
    final_map50 = map50_list[-1]
    final_params = base_nparams * nparams_list[-1] / 100

    all_results[f"{TARGET_PRUNE_RATE:.1f}"] = {
        "target_prune_rate": TARGET_PRUNE_RATE,
        "baseline_nparams_M": base_nparams / 1e6,
        "final_nparams_M": final_params / 1e6,
        "param_reduction_percent": 100 - nparams_list[-1],
        "baseline_map50_95": init_map,
        "final_map50_95": final_map,
        "baseline_map50": init_map50,
        "final_map50": final_map50,
        "map_loss_percent": (init_map - final_map) * 100,
        "baseline_macs_G": base_macs / 1e9,
        "final_macs_G": pruned_macs / 1e9,
        "speedup": float(macs_list[0]) / pruned_macs,
        "model_path": str(model.trainer.best),
    }

    elapsed = time.time() - overall_start_time
    print(f"\n⏱️  Total elapsed: {elapsed/3600:.2f} h")


results_root = Path("single_shot_pruned_v5")
results_root.mkdir(parents=True, exist_ok=True)

results_json_path = results_root / "results_summary.json"
with open(results_json_path, "w") as f:
    json.dump(all_results, f, indent=4)
print(f"\n✅ Results saved to {results_json_path}")


c:\Users\alijo\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful!
PyTorch: 2.6.0+cu124
CUDA available: True
torch_pruning version: 1.6.0
✅ torch.load patched for PyTorch 2.6+ compatibility
✅ Training override functions defined!
MULTI-RATIO PRUNING EXPERIMENT (YOLOv5n/u - C3)
Target pruning ratios: [0.047]
Iterative steps per ratio: 1
Total experiments: 1

RATIO 1/1: TARGET PRUNE RATE = 5%
Loading model...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\alijo\\OneDrive\\Desktop\\BachelorThesis\\BachelorThesis\\Models\\runs_yolov5\\leafminer_yolov5n_ultralytics\\weights\\best.pt'

BN prunning

In [ ]:
# =========================================================
# YOLOv5 Iterative (Geometric per-step) Structured Pruning
# =========================================================

import math
import time
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch_pruning as tp

from ultralytics import YOLO
from ultralytics.nn.modules import Detect


# -----------------------------
# Helpers
# -----------------------------
def torch_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

def get_detect_ignored_submodules_yolov5_safe(yolo_nn: nn.Module):
    """
    YOLOv5 in ultralytics typically has Detect with attribute:
      - m: nn.ModuleList of output convs (one per detection scale)
    YOLOv8 Detect has cv2/cv3/dfl, etc.

    We DO NOT ignore Detect itself. We ignore only the final prediction conv(s).
    """
    detect = None
    for m in yolo_nn.modules():
        if isinstance(m, Detect):
            detect = m
            break

    ignored = []
    if detect is not None:
        if hasattr(detect, "m"):
            mod = getattr(detect, "m")
            if isinstance(mod, (list, nn.ModuleList)):
                ignored += list(mod)
            else:
                ignored.append(mod)

        for attr in ["cv2", "cv3", "dfl"]:
            if hasattr(detect, attr):
                mod = getattr(detect, attr)
                if isinstance(mod, (list, nn.ModuleList)):
                    ignored += list(mod)
                else:
                    ignored.append(mod)

    # remove duplicates while preserving order
    seen = set()
    uniq = []
    for x in ignored:
        if id(x) not in seen:
            uniq.append(x)
            seen.add(id(x))

    return detect, uniq

def de_inference_parameters(model: nn.Module):
    """
    PyTorch 2.x fix: after pruning, some params can become "inference tensors".
    Ultralytics tries to set requires_grad=True and can crash.
    Convert inference tensors back to normal nn.Parameter.
    """
    for module in model.modules():
        for name, p in list(module._parameters.items()):
            if p is None:
                continue
            if hasattr(torch, "is_inference") and torch.is_inference(p):
                module._parameters[name] = nn.Parameter(p.detach().clone(), requires_grad=True)
            else:
                p.requires_grad_(True)

def load_best_into_current_pruned_model(yolo: YOLO, best_path: str, device: torch.device):
    """
    Load best.pt weights INTO the CURRENT (already pruned) yolo.model.
    Avoids YOLO(best.pt) which can rebuild a fresh unpruned architecture.
    """
    ckpt = torch.load(best_path, map_location="cpu")

    state = None
    if isinstance(ckpt, dict):
        if "ema" in ckpt and ckpt["ema"] is not None:
            state = ckpt["ema"].state_dict()
        elif "model" in ckpt and ckpt["model"] is not None:
            m = ckpt["model"]
            state = m.state_dict() if hasattr(m, "state_dict") else m
        elif "state_dict" in ckpt:
            state = ckpt["state_dict"]
    elif hasattr(ckpt, "state_dict"):
        state = ckpt.state_dict()

    if state is None:
        raise RuntimeError(f"Could not extract state_dict from: {best_path}")

    missing, unexpected = yolo.model.load_state_dict(state, strict=False)
    yolo.model.to(device).train()

    # ensure no inference tensors remain after loading
    de_inference_parameters(yolo.model)

    print(f"✅ Loaded best weights into PRUNED model: {best_path}")
    print(f"   Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")


# ========================================
# CONFIGURATION (YOLOv5)
# ========================================
MODEL_PATH = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Models\runs_yolov5\extra_10_epochs\weights\best.pt"
# (or use 'yolov5nu.pt' if you want the newer ultralytics-trained v5-u backbone)
DATA_YAML  = r"C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\MixedDS.yaml"

ITERATIVE_STEPS   = 1
TARGET_PRUNE_RATE = 0.68
MAX_MAP_DROP      = 1.0

pruning_cfg = {
    "data": DATA_YAML,
    "epochs": 100,     
    "batch": 16,
    "imgsz": 640,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "project": "YOLO_V5_BN_0.8",
    "exist_ok": True,
    "patience": 100,
    "save": True,
    "verbose": True,
    "plots": True,
    "name": "ONESHOT_PRUNING",
   
}

#steps: remaining = (1-r)^S = (1-T)
pruning_ratio = 1 - (1 - TARGET_PRUNE_RATE) ** (1 / ITERATIVE_STEPS)

print("=" * 70)
print("PRUNING CONFIGURATION (YOLOv5)")
print("=" * 70)
print(f"Model: {MODEL_PATH}")
print(f"Data: {DATA_YAML}")
print(f"Iterative steps: {ITERATIVE_STEPS}")
print(f"Target prune rate: {TARGET_PRUNE_RATE*100:.1f}%")
print(f"Per-step ratio: {pruning_ratio:.4f} ({pruning_ratio*100:.2f}%)")
print(f"Max mAP drop: {MAX_MAP_DROP}")
print(f"Fine-tune epochs: {pruning_cfg['epochs']}")
print(f"Device: {pruning_cfg['device']}")
print("=" * 70)

assert Path(MODEL_PATH).exists(), f"❌ Model not found: {MODEL_PATH}"
assert Path(DATA_YAML).exists(), f"❌ Data YAML not found: {DATA_YAML}"
print("✅ Model found")
print("✅ Data YAML found")



print("=" * 70)
print("LOADING MODEL")
print("=" * 70)

model = YOLO(MODEL_PATH)
model.__setattr__("train_v2", train_v2.__get__(model))  

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
pruning_cfg["device"] = 0 if device.type == "cuda" else "cpu"

model.model.to(device).train()
de_inference_parameters(model.model)
print(f"✅ Model loaded on {device}")

example_inputs = torch.randn(1, 3, pruning_cfg["imgsz"], pruning_cfg["imgsz"]).to(device)

base_macs, base_nparams = tp.utils.count_ops_and_params(model.model, example_inputs)
print("\n📊 Baseline Model:")
print(f"  MACs: {base_macs/1e9:.3f} G")
print(f"  Params(tp): {base_nparams/1e6:.3f} M")
print(f"  Params(torch): {torch_params(model.model)/1e6:.3f} M")


print("=" * 70)
print("BASELINE VALIDATION")
print("=" * 70)

macs_list, nparams_list, map_list, map50_list = [], [], [], []

pruning_cfg["name"] = "baseline_val"
metric = deepcopy(model).val(**pruning_cfg)
init_map = float(metric.box.map)
init_map50 = float(metric.box.map50)

macs_list.append(float(base_macs))
nparams_list.append(100.0)
map_list.append(init_map)
map50_list.append(init_map50)

print("\n✅ Baseline Results:")
print(f"  mAP50-95: {init_map:.4f}")
print(f"  mAP50:    {init_map50:.4f}")

print("\n" + "=" * 70)
print("⚠️  STARTING ITERATIVE PRUNING (YOLOv5 + BNScale)")
print("=" * 70)
print(f"{ITERATIVE_STEPS} iterations × {pruning_cfg['epochs']} epochs")
print("=" * 70 + "\n")

start_time = time.time()

for i in range(ITERATIVE_STEPS):
    print("\n" + "=" * 70)
    print(f"ITERATION {i+1}/{ITERATIVE_STEPS}")
    print("=" * 70)

    model.model.to(device).train()
    de_inference_parameters(model.model)

    detect, ignored_layers = get_detect_ignored_submodules_yolov5_safe(model.model)
    print(f"Detect found: {detect is not None} | Ignored submodules: {len(ignored_layers)}")

    pruner = tp.pruner.BNScalePruner(
        model.model,
        example_inputs,
        importance=tp.importance.BNScaleImportance(),
        pruning_ratio=pruning_ratio,
        ignored_layers=ignored_layers,
        global_pruning=True,
    )

    before = torch_params(model.model)
    print(f"Pruning {pruning_ratio*100:.2f}% of channels (this step)...")

    with torch.inference_mode(False):
        pruner.step()

    de_inference_parameters(model.model)

    after = torch_params(model.model)
    print(f"torch params: {before/1e6:.3f}M -> {after/1e6:.3f}M (delta {(before-after)/1e6:.3f}M)")

    pruning_cfg["name"] = f"step_{i}_pre_val"
    metric = model.val(**pruning_cfg)
    pruned_map = float(metric.box.map)

    pruned_macs, pruned_nparams = tp.utils.count_ops_and_params(model.model, example_inputs)
    speedup = float(base_macs) / float(pruned_macs)

    print("\n📊 After Pruning:")
    print(f"  MACs: {pruned_macs/1e9:.3f} G (speedup: {speedup:.2f}x)")
    print(f"  Params: {pruned_nparams/1e6:.3f} M ({(pruned_nparams/base_nparams)*100:.1f}% remaining)")
    print(f"  mAP50-95 (pre-FT): {pruned_map:.4f}")

    print(f"\n🔧 Fine-tuning for {pruning_cfg['epochs']} epochs...")
    pruning_cfg["name"] = f"step_{i}_finetune"

    de_inference_parameters(model.model)
    model.train_v2(pruning=True, **pruning_cfg)

    best_path = model.trainer.best
    load_best_into_current_pruned_model(model, best_path, device)

    pruning_cfg["name"] = f"step_{i}_post_val"
    metric = model.val(**pruning_cfg)
    current_map = float(metric.box.map)
    current_map50 = float(metric.box.map50)

    post_macs, post_nparams = tp.utils.count_ops_and_params(model.model, example_inputs)
    post_speedup = float(base_macs) / float(post_macs)

    print("\n After Fine-tuning:")
    print(f"  Params(tp): {post_nparams/1e6:.3f} M ({(post_nparams/base_nparams)*100:.1f}% remaining)")
    print(f"  MACs: {post_macs/1e9:.3f} G (speedup: {post_speedup:.2f}x)")
    print(f"  mAP50-95: {current_map:.4f}")
    print(f"  mAP50:    {current_map50:.4f}")
    print(f"  POST-FT torch params: {torch_params(model.model)/1e6:.3f} M")

    macs_list.append(float(post_macs))
    nparams_list.append(float(post_nparams / base_nparams * 100))
    map_list.append(current_map)
    map50_list.append(current_map50)

    save_pruning_performance_graph(nparams_list, map_list, map50_list, macs_list, "BNScale_YOLOv5")

    if (init_map - current_map) > MAX_MAP_DROP:
        print(f"\n⚠️ Early stopping! mAP drop ({init_map - current_map:.4f}) exceeds threshold ({MAX_MAP_DROP})")
        break

print("\n" + "=" * 70)
print("✅ PRUNING COMPLETE!")
print("=" * 70)
total_time = time.time() - start_time
print(f"Total time: {total_time/3600:.2f} hours")


print("=" * 70)
print("FINAL PRUNING RESULTS")
print("=" * 70)

print(f"{'Step':<6} {'Params %':<12} {'mAP50-95':<12} {'mAP50':<12}")
print("-" * 45)
for idx, (params_pct, map_val, map50_val) in enumerate(zip(nparams_list, map_list, map50_list)):
    print(f"{idx:<6} {params_pct:<12.1f} {map_val:<12.4f} {map50_val:<12.4f}")

best_idx = int(np.argmax(map_list))
print(f"\n🏆 Best Result:")
print(f"  Step: {best_idx}")
print(f"  Params: {nparams_list[best_idx]:.1f}% ({base_nparams*nparams_list[best_idx]/100/1e6:.2f}M)")
print(f"  mAP50-95: {map_list[best_idx]:.4f}")
print(f"  mAP50:    {map50_list[best_idx]:.4f}")



print("=" * 70)
print("EXPORTING FINAL PRUNED MODEL")
print("=" * 70)

try:
    best_path = model.trainer.best
    load_best_into_current_pruned_model(model, best_path, device)

    print("\nExporting to ONNX...")
    onnx_path = model.export(format="onnx", imgsz=pruning_cfg["imgsz"], simplify=True)
    print(f"\n✅ ONNX exported: {onnx_path}")
    print(f"File size: {Path(onnx_path).stat().st_size / 1e6:.2f} MB")
except Exception as e:
    print(f"❌ Export failed: {e}")

print("\n🎉 ALL DONE!")


PRUNING CONFIGURATION (YOLOv5)
Model: C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\Models\runs_yolov5\extra_10_epochs\weights\best.pt
Data: C:\Users\alijo\OneDrive\Desktop\BachelorThesis\BachelorThesis\MixedDS.yaml
Iterative steps: 1
Target prune rate: 68.0%
Per-step ratio: 0.6800 (68.00%)
Max mAP drop: 1.0
Fine-tune epochs: 100
Device: 0
✅ Model found
✅ Data YAML found
LOADING MODEL
✅ Model loaded on cuda:0

📊 Baseline Model:
  MACs: 3.561 G
  Params(tp): 2.509 M
  Params(torch): 2.509 M
BASELINE VALIDATION
Ultralytics 8.3.223  Python-3.10.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLOv5n summary (fused): 84 layers, 2,503,139 parameters, 12,691 gradients, 7.1 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 27.45.3 MB/s, size: 39.7 KB)
val: Scanning C:\Users\alijo\OneDrive\Desktop\LeafminerOnly\val\labels.cache... 314 images, 106 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 314/314 316.1Kit/s 0.0s
WARNING Box and segment counts shoul

NameError: name 'save_pruning_performance_graph' is not defined